In [1]:
import os
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import train_test_split
from PIL import Image
from tqdm import tqdm

# ==========================================
# تنظیمات اولیه پروژه
# ==========================================

print("start Vision Transformer")

DATA_PATH = "D:/PhD Courses/DNN/finalProject/data"     # مسیر دیتاست
IMG_SIZE = 224         # اندازه ورودی تصویر
BATCH_SIZE = 32        # اندازه هر batch
EPOCHS = 20            # تعداد epoch
LR = 3e-4              # نرخ یادگیری

# بررسی GPU
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"{DEVICE}")

torch.backends.cudnn.benchmark = True


# ==========================================
# Dataset
# ==========================================

class DriverDataset(Dataset):
    def __init__(self, data_dir, transform=None):
        print("reading data")
        self.images = []
        self.labels = []
        self.transform = transform

        # خواندن پوشه‌های کلاس
        for class_folder in sorted(os.listdir(data_dir)):
            class_path = os.path.join(data_dir, class_folder)
            if os.path.isdir(class_path):

                # استخراج لیبل (مثلاً c3 → 3)
                label = int(class_folder.replace("c", ""))

                for img in os.listdir(class_path):
                    if img.endswith((".jpg", ".png", ".jpeg")):
                        self.images.append(os.path.join(class_path, img))
                        self.labels.append(label)

        print(f" number of pic: {len(self.images)}")

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = Image.open(self.images[idx]).convert("RGB")
        label = self.labels[idx]

        # اعمال پیش‌پردازش
        if self.transform:
            img = self.transform(img)

        return img, label


# ==========================================
# preprocessing
# ==========================================

print("transforms...")

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),     # تغییر سایز
    transforms.RandomHorizontalFlip(0.5),        # افزایش تنوع داده
    transforms.ColorJitter(0.2, 0.2, 0.2, 0.05), # تغییر نور و رنگ
    transforms.ToTensor(),                       # تبدیل به Tensor
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])    # نرمال‌سازی
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])


# ==========================================
# train and validation
# ==========================================

full_dataset = DriverDataset(DATA_PATH, transform=train_transform)

print("devide data to Train and Validation...")

train_idx, val_idx = train_test_split(
    range(len(full_dataset)),
    test_size=0.2,
    stratify=full_dataset.labels,
    random_state=42
)

train_dataset = Subset(full_dataset, train_idx)
val_dataset = Subset(full_dataset, val_idx)

# change transform for validation
val_dataset.dataset.transform = val_transform

train_loader = DataLoader(train_dataset,
                          batch_size=BATCH_SIZE,
                          shuffle=True,
                          pin_memory=True)

val_loader = DataLoader(val_dataset,
                        batch_size=BATCH_SIZE,
                        shuffle=False,
                        pin_memory=True)

print("dataloaders are ready")


# ==========================================
#  Vision Transformer
# ==========================================

print("Vision Transformer model...")

class PatchEmbedding(nn.Module):
    def __init__(self, img_size=224, patch_size=16,
                 in_ch=3, embed_dim=512):
        super().__init__()
        # Conv2D برای تبدیل patch ها به embedding
        self.proj = nn.Conv2d(in_ch, embed_dim,
                              kernel_size=patch_size,
                              stride=patch_size)

    def forward(self, x):
        x = self.proj(x)
        x = x.flatten(2)
        x = x.transpose(1, 2)
        return x


class TransformerBlock(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = nn.MultiheadAttention(embed_dim,
                                          num_heads,
                                          batch_first=True)
        self.norm2 = nn.LayerNorm(embed_dim)

        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, embed_dim*4),
            nn.GELU(),
            nn.Linear(embed_dim*4, embed_dim)
        )

    def forward(self, x):
        # Self-Attention + Residual
        x = x + self.attn(self.norm1(x),
                          self.norm1(x),
                          self.norm1(x))[0]

        # MLP + Residual
        x = x + self.mlp(self.norm2(x))
        return x


class VisionTransformer(nn.Module):
    def __init__(self,
                 img_size=224,
                 patch_size=16,
                 embed_dim=512,
                 depth=6,
                 num_heads=8,
                 num_classes=10):

        super().__init__()

        self.patch_embed = PatchEmbedding(img_size,
                                          patch_size,
                                          3,
                                          embed_dim)

        num_patches = (img_size//patch_size)**2

        # CLS token
        self.cls_token = nn.Parameter(torch.zeros(1,1,embed_dim))

        # Positional embedding
        self.pos_embed = nn.Parameter(torch.zeros(1,
                                                  num_patches+1,
                                                  embed_dim))

        # Transformer blocks
        self.blocks = nn.ModuleList([
            TransformerBlock(embed_dim, num_heads)
            for _ in range(depth)
        ])

        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, num_classes)

    def forward(self, x):

        B = x.shape[0]

        # تبدیل تصویر به patch embedding
        x = self.patch_embed(x)

        # اضافه کردن CLS token
        cls_tokens = self.cls_token.expand(B,-1,-1)
        x = torch.cat((cls_tokens, x), dim=1)

        # اضافه کردن موقعیت مکانی
        x = x + self.pos_embed

        # عبور از بلاک‌های Transformer
        for block in self.blocks:
            x = block(x)

        x = self.norm(x)

        # فقط خروجی CLS برای طبقه‌بندی
        return self.head(x[:,0])


# ==========================================
# tarining
# ==========================================

model = VisionTransformer().to(DEVICE)
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(model.parameters(),
                              lr=LR,
                              weight_decay=0.01)

print("begin to train...")




def evaluate(loader):
    model.eval()
    correct, total, loss_sum = 0, 0, 0

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            outputs = model(x)
            loss = criterion(outputs, y)

            loss_sum += loss.item()
            preds = torch.argmax(outputs, 1)
            correct += (preds == y).sum().item()
            total += y.size(0)

    return loss_sum/len(loader), correct/total


for epoch in range(EPOCHS):

    print(f"\n Epoch {epoch+1}/{EPOCHS}")

    model.train()
    running_loss, correct, total = 0, 0, 0

    for x, y in tqdm(train_loader):
        x, y = x.to(DEVICE), y.to(DEVICE)

        optimizer.zero_grad()

        outputs = model(x)
        loss = criterion(outputs, y)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        preds = torch.argmax(outputs, 1)
        correct += (preds == y).sum().item()
        total += y.size(0)

    train_loss = running_loss/len(train_loader)
    train_acc = correct/total

    val_loss, val_acc = evaluate(val_loader)

    print(f"Train Loss: {train_loss:.4f} | Acc: {train_acc:.4f}")
    print(f"Val   Loss: {val_loss:.4f} | Acc: {val_acc:.4f}")

print("train is complete")

# ==========================================
# Confusion Matrix
# ==========================================

print("Confusion Matrix")

from sklearn.metrics import confusion_matrix
import seaborn as sns

model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for x, y in val_loader:
        x = x.to(DEVICE)
        outputs = model(x)
        preds = torch.argmax(outputs, 1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y.numpy())

# محاسبه ماتریس
cm = confusion_matrix(all_labels, all_preds)

print("Confusion Matrix :")
print(cm)


plt.figure(figsize=(8,6))
sns.heatmap(cm,
            annot=True,
            fmt="d",
            cmap="Blues",
            xticklabels=[f"Class {i}" for i in range(10)],
            yticklabels=[f"Class {i}" for i in range(10)])

plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix")
plt.show()

start Vision Transformer
cuda
transforms...
reading data
✅ number of pic: 3868
devide data to Train and Validation...
dataloaders are ready
Vision Transformer model...
begin to train...

 Epoch 1/20


 12%|█▏        | 12/97 [00:12<01:27,  1.02s/it]


KeyboardInterrupt: 